# ViT-Tiny on CIFAR-10

Implementation of Vision Transformer (Dosovitskiy et al., 2021) from scratch on CIFAR-10.

**Hyperparameters (ViT-Tiny) :**
- Image size : 32×32, 3 channels
- Patch size : P = 4 → N = 64 patches
- T = N + 1 = 65 tokens (patches + CLS)
- d_model = 128
- h = 4
- d_k = 32
- num_layers = 6 blocks
- d_ff = 4 × d_model = 512
- num_classes = 10

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import math

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"device = {device}")

---
## 0. Dataset

## Data transforms

### Training set

| Transform | Description |
|---|---|
| `RandomHorizontalFlip()` | Flips the image horizontally with probability 0.5 : forces the model to be invariant to left/right orientation |
| `RandomCrop(32, padding=4)` | Pads the image with 4 black pixels on each side, then randomly crops back to 32×32 — simulates slight translations |
| `ToTensor()` | Converts a PIL image (H, W, C) with values in [0, 255] to a PyTorch tensor (C, H, W) with values in [0.0, 1.0] |
| `Normalize(mean, std)` | Standardizes each channel : $x \leftarrow \frac{x - \mu}{\sigma}$ — centers the data around 0, stabilizes training |

CIFAR-10 statistics used for normalization (computed over the full training set) :

| Channel | Mean | Std |
|---|---|---|
| R | 0.4914 | 0.2470 |
| G | 0.4822 | 0.2435 |
| B | 0.4465 | 0.2616 |

`RandomHorizontalFlip` and `RandomCrop` are **data augmentation** techniques :
they artificially increase the diversity of the training set without collecting new data,
which reduces overfitting.

### Test set

Only `ToTensor` and `Normalize` are applied - no augmentation.
Test images must not be randomly modified, otherwise the evaluation is not reproducible.

In [ ]:
BATCH_SIZE = 128

transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),         #mean
                         (0.2470, 0.2435, 0.2616)),        #std 
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616)),
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                              download=True, transform=transform_train)
test_dataset  = torchvision.datasets.CIFAR10(root='./data', train=False,
                                              download=True, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

sample_img, _ = train_dataset[0]
CHANNELS, H, W  = sample_img.shape
IMG_SIZE = H
NUM_CLASSES = len(train_dataset.classes)

print(f"Train size  : {len(train_dataset)}")
print(f"Test size   : {len(test_dataset)}")
print(f"Image size  : {CHANNELS} x {H} x {W}")
print(f"Num classes : {NUM_CLASSES}  {train_dataset.classes}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

classes = test_dataset.classes
indices = torch.randint(0, len(test_dataset), (10,))

fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for i, idx in enumerate(indices):
    img, label = test_dataset[idx]
    mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
    std  = torch.tensor([0.2470, 0.2435, 0.2616]).view(3, 1, 1)
    img = img * std + mean
    img = img.permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)

    axes[i].imshow(img)
    axes[i].set_title(classes[label], fontsize=8)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Hyperparameters  
PATCH_SIZE  = 4
N_PATCHES   = (IMG_SIZE // PATCH_SIZE) ** 2   # 64
T           = N_PATCHES + 1                    # 65  (+ CLS token)
D_MODEL     = 128
NUM_HEADS   = 4
D_FF        = 4 * D_MODEL                      # 512
NUM_LAYERS  = 6
DROPOUT     = 0.1

PATCH_DIM   = PATCH_SIZE ** 2 * CHANNELS    # 48  (flattened patch size)

print(f"N_PATCHES = {N_PATCHES}")
print(f"T         = {T}")
print(f"PATCH_DIM = {PATCH_DIM}")

---
## 1. Patch Embedding

Starting from a batch of images $X \in \mathbb{R}^{B \times C \times H \times W}$, we build $z_0 \in \mathbb{R}^{B \times T \times d_{\text{model}}}$ in 4 steps.


**Step 1 : Split into patches**

Each image is divided into $N = \frac{H}{P} \times \frac{W}{P}$ non-overlapping patches of size $P \times P$.
Each patch is then flattened into a vector of $P^2 \cdot C$ values.

$$X \in \mathbb{R}^{B \times C \times H \times W} \;\longrightarrow\; X_p \in \mathbb{R}^{B \times N \times (P^2 \cdot C)}$$

For CIFAR-10 with $P=4$ : $N = 8 \times 8 = 64$ patches, $P^2 \cdot C = 48$ values per patch.


**Step 2 : Linear projection**

Each patch is projected from raw pixel space to the model dimension $d_{\text{model}}$ using a learnable matrix $E \in \mathbb{R}^{(P^2 \cdot C) \times d_{\text{model}}}$ (`nn.Linear(48, 128)`).

$$X_p \in \mathbb{R}^{B \times N \times 48} \;\longrightarrow\; X_p E \in \mathbb{R}^{B \times N \times d_{\text{model}}}$$


**Step 3 : Prepend CLS token**

A learnable vector $x_{\text{class}} \in \mathbb{R}^{d_{\text{model}}}$ is prepended to the sequence as row 0.
It will aggregate global information about the image through the attention layers.

$$\mathbb{R}^{B \times N \times d_{\text{model}}} \;\longrightarrow\; \mathbb{R}^{B \times (N+1) \times d_{\text{model}}} = \mathbb{R}^{B \times T \times d_{\text{model}}}$$


**Step 4 : Add positional embeddings**

A learnable matrix $E_{\text{pos}} \in \mathbb{R}^{T \times d_{\text{model}}}$ is added to the sequence.
It encodes the position of each token, since self-attention is permutation-invariant.

$$z_0 = \begin{bmatrix} x_{\text{class}} \\ x_p^1 E \\ x_p^2 E \\ \vdots \\ x_p^N E \end{bmatrix} + E_{\text{pos}} \quad \in \mathbb{R}^{B \times T \times d_{\text{model}}}$$

where :
- $x_{\text{class}} \in \mathbb{R}^{d_{\text{model}}}$ : learnable CLS token
- $x_p^i \in \mathbb{R}^{P^2 \cdot C}$ : flattened patch $i$
- $E \in \mathbb{R}^{(P^2 \cdot C) \times d_{\text{model}}}$ : learnable projection matrix (`nn.Linear(48, 128)`)
- $E_{\text{pos}} \in \mathbb{R}^{T \times d_{\text{model}}}$ : learnable positional embeddings

For CIFAR-10 : $z_0 \in \mathbb{R}^{B \times 65 \times 128}$.

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, patch_size, channels, d_model, n_patches, dropout=0.1): #4,3,128,64
        super().__init__()
        
        patch_dim = patch_size ** 2 * channels   # 48
        self.projection  = nn.Linear(patch_dim, d_model)   #E(patch_dim,d_model)
        self.cls_token   = nn.Parameter(torch.randn(1, 1, d_model))   #x_class dim 1,1,d_model
        self.pos_embedding = nn.Parameter(torch.randn(1, n_patches + 1, d_model))   #Epos(1,T,d_model)

        self.dropout = nn.Dropout(dropout)
        self.patch_size = patch_size #4

    def forward(self, x):
        # x : (B, C, H, W)
        B, C, H, W = x.shape
        P = self.patch_size #4
        x = x.reshape(B, C, H//P, P, W//P, P)  # (B, C, H, W) -> (B, C, H//P, P, W//P, P)
        x = x.permute(0, 2, 4, 3, 5, 1)  # (B, C, H//P, P, W//P, P) -> (B, H//P, W//P, P, P, C)
        x = x.reshape(B, (H//P) * (W//P), P * P * C)   # # (B, H//P, W//P, P, P, C) -> (B, N_PATCHES, patch_dim) = (B, 64, 48)

        x = self.projection(x)                          # (B, N_PATCHES, d_model)

        # prepend CLS token
        cls_tokens = self.cls_token.expand(B, -1, -1)  # (B, 1, 128)
        x = torch.cat([cls_tokens, x], dim=1)           # (B, 65, 128)

        # add positional embeddings
        x = x + self.pos_embedding                      # (B, 65, 128)

        return self.dropout(x)

In [ ]:
patch_embed = PatchEmbedding(PATCH_SIZE, CHANNELS, D_MODEL, N_PATCHES, dropout=0.0).to(device)
x_test = torch.randn(BATCH_SIZE, CHANNELS, IMG_SIZE, IMG_SIZE).to(device)
out = patch_embed(x_test)
print(f"PatchEmbedding output shape : {out.shape}")   # expected : (128, 65, 128)

---
## 2. ViT

Assembles `PatchEmbedding` + `TransformerEncoder` + classification head into the full ViT pipeline.

**Forward pass :**

| Step | Operation | Shape |
|---|---|---|
| Input | image batch | $(B, C, H, W)$ |
| Patch embedding | `PatchEmbedding` | $(B, T, d_{\text{model}})$ → $z_0$ |
| Transformer encoder | $L$ blocks of MSA + MLP (Pre-LN) | $(B, T, d_{\text{model}})$ → $z_L$ |
| CLS token | extract row 0 : `x[:, 0, :]` | $(B, d_{\text{model}})$ |
| Classification head | `LayerNorm` + `Linear` | $(B, \text{num\_classes})$ |

Each Transformer block applies :
$$z'_\ell = \text{MSA}(\text{LN}(z_{\ell-1})) + z_{\ell-1}$$
$$z_\ell = \text{MLP}(\text{LN}(z'_\ell)) + z'_\ell$$

In [ ]:
class ViT(nn.Module):
    def __init__(self, img_size, patch_size, in_channels, d_model, num_heads,
                 num_layers, d_ff, num_classes, dropout=0.1):
        
        super().__init__()
        
        n_patches = (img_size // patch_size) ** 2

        self.patch_embedding = PatchEmbedding(patch_size, in_channels, d_model, n_patches, dropout)

        block = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,    # Pre-LN (ViT style)
        )
        
        self.encoder = nn.TransformerEncoder(block, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, x):
        # x : (B, C, H, W)
        x = self.patch_embedding(x)   # z0(B, T, d_model)
        x = self.encoder(x)           # zL(B, T, d_model)
        x = x[:, 0, :]               # (B, d_model)  — CLS token (row 0)
        x = self.norm(x)              # (B, d_model)
        x = self.head(x)              # (B, num_classes)
        return x

In [ ]:
model = ViT(
    img_size=IMG_SIZE,
    patch_size=PATCH_SIZE,
    in_channels=CHANNELS,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    d_ff=D_FF,
    num_classes=NUM_CLASSES,
    dropout=0.0
).to(device)

logits = model(x_test)
print(f"ViT output shape : {logits.shape}")  
print(f"n_params : {sum(p.numel() for p in model.parameters()):,}")

---
## 3. Training

**Optimizer :** Adam with lr = 1e-3

**Loss :** CrossEntropyLoss — computes softmax + negative log-likelihood in one step :
$$\mathcal{L} = -\log \frac{e^{z_y}}{\sum_k e^{z_k}}$$
where $z_y$ is the logit of the correct class.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += images.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        logits = model(images)
        loss   = criterion(logits, labels)

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += images.size(0)

    return total_loss / total, correct / total

In [ ]:
NUM_EPOCHS = 30
LR         = 1e-3

model = ViT(
    img_size=IMG_SIZE,
    patch_size=PATCH_SIZE,
    in_channels=CHANNELS,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    d_ff=D_FF,
    num_classes=NUM_CLASSES,
    dropout=DROPOUT
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

print(f"n_params : {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
train_losses, test_losses = [], []
train_accs,   test_accs   = [], []

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    test_loss,  test_acc  = evaluate(model, test_loader, criterion)

    train_losses.append(train_loss)
    test_losses.append(test_loss)
    train_accs.append(train_acc)
    test_accs.append(test_acc)

    print(f"Epoch {epoch:2d}/{NUM_EPOCHS} | "
          f"train loss {train_loss:.4f}  train acc {train_acc*100:.1f}%  | "
          f"test loss {test_loss:.4f}  test acc {test_acc*100:.1f}%")

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, train_losses, label='train')
ax1.plot(epochs, test_losses,  label='test')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss')
ax1.legend()

ax2.plot(epochs, [a*100 for a in train_accs], label='train')
ax2.plot(epochs, [a*100 for a in test_accs],  label='test')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Accuracy')
ax2.legend()

plt.tight_layout()
plt.show()

print(f"Best test accuracy : {max(test_accs)*100:.1f}%")

In [ ]:
model.eval()
indices = torch.randint(0, len(test_dataset), (10,))
mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
std  = torch.tensor([0.2470, 0.2435, 0.2616]).view(3, 1, 1)

fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for i, idx in enumerate(indices):
    img, true_label = test_dataset[idx]
    
    with torch.no_grad():
        logits = model(img.unsqueeze(0).to(device))
        pred_label = logits.argmax(dim=1).item()

    img_display = img * std + mean
    img_display = img_display.permute(1, 2, 0).numpy()
    img_display = np.clip(img_display, 0, 1)

    axes[i].imshow(img_display)
    color = 'green' if pred_label == true_label else 'red'
    axes[i].set_title(f"pred: {classes[pred_label]}\ntrue: {classes[true_label]}", fontsize=6, color=color)
    axes[i].axis('off')

plt.tight_layout()
plt.show()